# Gemini email-agent judge pilot
Run on Kaggle with the dataset attached and `GEMINI_API_KEY` in Kaggle Secrets. Push the other `llm_judge/` files to GitHub before running this notebook.


In [ ]:
!pip -q install google-genai pandas openpyxl scikit-learn


In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path("/kaggle/working/AI_Gaurdrails")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/aarohichadha/AI_Gaurdrails.git", str(repo)], check=True)
sys.path.insert(0, str(repo))
from llm_judge.judge import GeminiJudge, METHODS
from llm_judge.evaluation import load_dataset, paired_pilot, score, destination_oracle_rate
from kaggle_secrets import UserSecretsClient
import pandas as pd


In [ ]:
candidates = sorted(Path("/kaggle/input").rglob("email_agent_security_dataset.xlsx"))
if not candidates:
    candidates = [repo / "data/email_agent_security_dataset.xlsx"]
path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError("Attach email_agent_security_dataset.xlsx")
data = load_dataset(path)
pilot = paired_pilot(data, n_pairs=100)
print("Development records:", len(pilot))
print("Destination-only rule accuracy:", destination_oracle_rate(pilot))


## Run methods
Set `METHODS_TO_RUN = ("plain",)` for a first run. Gemini API requests may incur charges.


In [ ]:
key = UserSecretsClient().get_secret("GEMINI_API_KEY")
if not key:
    raise RuntimeError("Set Kaggle secret GEMINI_API_KEY")
judge = GeminiJudge(key)
METHODS_TO_RUN = METHODS
output_dir = Path("/kaggle/working/llm_judge_results")
output_dir.mkdir(parents=True, exist_ok=True)
all_results = []
for method in METHODS_TO_RUN:
    rows = []
    for _, row in pilot.iterrows():
        answer = judge.judge(row, method)
        rows.append({"record_id": row.record_id, "pair_id": row.pair_id, "label": row.label,
                     "expected_decision": row.expected_decision, "method": method, **answer})
    pd.DataFrame(rows).to_json(output_dir / f"{method}.jsonl", orient="records", lines=True)
    all_results += rows
    print(method, score(rows))


In [ ]:
results = pd.DataFrame(all_results)
summary = pd.DataFrame([{"method": method, **score(group.to_dict("records"))}
                        for method, group in results.groupby("method")])
display(summary)
display(results[results.decision.ne(results.expected_decision)].head(20))
summary.to_csv(output_dir / "summary.csv", index=False)
print("Files:", output_dir)
